# Advanced Prompt Engineering Techniques

# Advanced Prompt Engineering Techniques

Welcome to the hands-on exploration of advanced prompt engineering! In this notebook, you'll see these techniques in action and understand why prompt engineering is an **empirical science** that requires experimentation and iteration.

## What You'll Learn

- Core prompting techniques (zero-shot, few-shot, one-shot, role, emotional, chain-of-thought)
- Why LLM outputs are non-deterministic and how to control consistency
- Automatic prompt generation and meta-prompting strategies
- Token optimization techniques to reduce costs
- Model biases and how they affect outputs
- Using LLMs as judges to evaluate responses

## Prerequisites

Before starting, make sure you have:
- OpenAI API key set up
- Basic understanding of prompt engineering fundamentals (see lesson m2_01)
- Python environment with openai library installed

Let's dive in!

## 1. Setup & Configuration

First, let's import the necessary libraries and configure our OpenAI client.

In [ ]:
import os
import json
import time
from collections import Counter
from dataclasses import dataclass
from typing import Optional

from dotenv import load_dotenv
from openai import OpenAI


@dataclass(frozen=True)
class NotebookConfig:
    """Reusable notebook configuration loaded from environment variables."""

    model: str = "gpt-4o-mini"
    api_key_env: str = "OPENAI_API_KEY"
    env_file: Optional[str] = None

    @classmethod
    def from_env(cls, *, model: Optional[str] = None, env_file: Optional[str] = None) -> "NotebookConfig":
        """Load environment variables and build a validated config object."""
        load_dotenv(dotenv_path=env_file)
        return cls(model=model or os.getenv("OPENAI_MODEL", cls.model), env_file=env_file)

    @property
    def api_key(self) -> str:
        return os.getenv(self.api_key_env, "").strip()

    def require_api_key(self) -> str:
        if not self.api_key:
            raise ValueError(f"{self.api_key_env} is missing. Add it to your environment or .env file.")
        return self.api_key

    def create_client(self) -> OpenAI:
        return OpenAI(api_key=self.require_api_key())


config = NotebookConfig.from_env()
client = config.create_client()

# Keep MODEL available for the existing helper functions below.
MODEL = config.model

print(f"Setup complete! OpenAI client initialized with {MODEL}.")

### Helper Functions

Let's create some utility functions to make our experiments easier to manage and visualize.

In [2]:
def call_openai(prompt, system_message="You are a helpful assistant.", temperature=0.7, max_tokens=None, seed=None):
    """Helper function to call OpenAI API with specified parameters"""
    try:
        params = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature
        }
        
        if max_tokens:
            params["max_tokens"] = max_tokens
        if seed is not None:
            params["seed"] = seed
            
        response = client.chat.completions.create(**params)
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

def display_response(title, response, params=None):
    """Display a response with formatting"""
    print(f"\n{'='*60}")
    print(f"ðŸ“ {title}")
    if params:
        print(f"Parameters: {params}")
    print(f"{'='*60}")
    print(response)
    print(f"{'='*60}\n")

def count_tokens_approx(text):
    """Approximate token count (roughly 4 characters per token)"""
    return len(text) // 4

print("âœ… Helper functions loaded!")

âœ… Helper functions loaded!


## 2. Core Prompting Techniques

Let's explore the fundamental prompting techniques you learned about in the lesson. Each example demonstrates when and how to use these techniques effectively.

### Zero-Shot Prompting

Zero-shot prompting provides a clear, detailed description of what you need **without examples**. This is the most commonly used prompting technique in practice.

In [3]:
# Zero-shot: Just a clear instruction
prompt = "Write a haiku about AI Consulting and Integration"

response = call_openai(prompt, temperature=0.7)
display_response("Zero-Shot Prompting", response)


ðŸ“ Zero-Shot Prompting
Machines learn and guide,  
Bridging gaps with seamless codeâ€”  
Future redefined.



### One-Shot Prompting

One-shot prompting provides **a single example** to demonstrate the expected output format. This helps the model understand the pattern you want.

In [4]:
"""Q: What is prompt engineering?
A: Prompt engineering is the practice of crafting effective input prompts to guide AI models toward desired outputs or behaviors.

Q: Why is prompt engineering useful?
A:
"""

response = call_openai(prompt, temperature=0.7)
display_response("One-Shot Prompting", response)


ðŸ“ One-Shot Prompting
Wisdom in code flows,  
Bridging minds with machine's grace,  
Future's path unfolds.



### Few-Shot Prompting

Few-shot prompting adds **multiple examples** of the desired output format to guide the model's response, helping it understand the pattern more clearly.

In [6]:
# Few-shot: Provide multiple examples
prompt = """
Follow th format:

Q: What is prompt engineering?
A: Prompt engineering is the practice of crafting effective input prompts to guide AI models toward desired outputs or behaviors.

Q: Why is prompt engineering useful?
A: Prompt engineering optimizes AI model outputs, enhancing accuracy, relevance, and task-specific performance.

Q: What is temperature in AI models?
A: Temperature controls the randomness of AI outputs, with lower values producing consistent results and higher values generating creative variations.

Q: What is chain-of-thought prompting?
A:"""

response = call_openai(prompt, temperature=0.7)
display_response("Few-Shot Prompting", response)


ðŸ“ Few-Shot Prompting
A: Chain-of-thought prompting is a technique where the prompt encourages the AI to reason through a problem step-by-step, improving its ability to arrive at more accurate and logical conclusions.



### Role Prompting

Role prompting asks the model to play a specific role or persona. Research shows it's more effective to define the assistant in **third person** ("they are a senior data scientist") rather than first person ("you are a senior data scientist").

In [ ]:
# Role prompting: Third person (more effective)
system_message = "The assistant is a senior data scientist with 15 years of experience in machine learning and statistical analysis." #as an AI Consultant #as an pirate
prompt = "Explain the bias-variance tradeoff in simple terms."

response = call_openai(prompt, system_message=system_message, temperature=0.7)
display_response("Role Prompting (Third Person)", response, {"role": "Senior Data Scientist"})


ðŸ“ Role Prompting (Third Person)
Parameters: {'role': 'Senior Data Scientist'}
The bias-variance tradeoff is a key concept in machine learning that helps us understand the sources of error in our models and how to balance them for better predictions.

1. **Bias**: This refers to the error introduced by approximating a real-world problem (which may be complex) with a simplified model. High bias means that the model is too simple and fails to capture the underlying patterns of the data, leading to systematic errors. For example, a linear model trying to fit a complex, non-linear relationship will have high bias and may underperform.

2. **Variance**: This refers to the error introduced by the model's sensitivity to small fluctuations in the training data. High variance means that the model captures noise or random fluctuations in the training data, which can lead to overfitting. An overfitted model performs well on training data but poorly on new, unseen data.

3. **Tradeoff**: The ke

### Emotional Prompting

Emotional prompting uses emotional stimuli to help the LLM better handle emotion and tone in its responses.

In [8]:
# Emotional prompting: Adding emotional context
prompt = """This is very important to my career. I need you to explain AI ethics in a way that will help me 
succeed in my upcoming presentation. Please be thorough and thoughtful."""

response = call_openai(prompt, temperature=0.7)
display_response("Emotional Prompting", response)


ðŸ“ Emotional Prompting
Absolutely, I'd be happy to help you understand AI ethics in a manner that will not only prepare you for your presentation but also provide you with a solid foundation for your career. Hereâ€™s a comprehensive overview:

### Introduction to AI Ethics

AI ethics is a field that explores the moral implications and societal impacts of artificial intelligence technologies. As AI systems become more integrated into various aspects of life, itâ€™s essential to consider their implications on privacy, fairness, accountability, and transparency.

### Key Principles of AI Ethics

1. **Fairness**:
   - **Bias and Discrimination**: AI systems can perpetuate or even exacerbate biases present in the data they are trained on. This can lead to unfair treatment of individuals based on race, gender, socioeconomic status, etc. Ensuring fairness means actively working to identify and mitigate these biases.
   - **Equity**: AI should be developed and deployed in a way that ensures

### Chain-of-Thought Prompting

Chain-of-thought provides **step-by-step reasoning instructions**, enabling models to tackle complex tasks through structured thinking. This technique gave origin to reasoning models.

In [9]:
# Chain-of-thought: Request step-by-step reasoning
prompt = """A store has 23 apples. They receive a shipment of 47 more apples, but 15 are bruised and must be discarded. 
They sell 38 apples. How many apples do they have left? 

Please solve this step-by-step, showing your reasoning at each stage."""

response = call_openai(prompt, temperature=0)
display_response("Chain-of-Thought Prompting", response)


ðŸ“ Chain-of-Thought Prompting
Let's break down the problem step-by-step.

1. **Initial Number of Apples**: The store starts with 23 apples.

2. **Shipment of Apples**: They receive a shipment of 47 more apples. 
   - To find the total number of apples after the shipment, we add the number of apples they had to the number of apples received:
   \[
   23 + 47 = 70
   \]
   So, after the shipment, the store has 70 apples.

3. **Bruised Apples**: Out of the 70 apples, 15 are bruised and must be discarded. 
   - To find the number of good apples left, we subtract the bruised apples from the total:
   \[
   70 - 15 = 55
   \]
   Now, the store has 55 good apples.

4. **Selling Apples**: The store sells 38 apples. 
   - To find out how many apples are left after the sale, we subtract the number of apples sold from the number of good apples:
   \[
   55 - 38 = 17
   \]
   Therefore, after selling 38 apples, the store has 17 apples left.

5. **Final Answer**: The store has **17 apples left**

## 3. Non-Deterministic Behavior Study

One of the most important concepts in prompt engineering is understanding that **LLM outputs are non-deterministic**. This means that running the same prompt multiple times can produce different results, even with identical parameters.

Let's explore this empirically!

### Part A: Demonstrating Inconsistency

We'll run the exact same prompt with identical parameters **10 times** to show how the outputs vary. This demonstrates why prompt engineering requires testing and iteration.

In [11]:
# Run the same prompt 10 times with temperature=0.7
prompt = "Instructions how to make coffee" #Write a haiku about AI
temperature = 0.7
num_runs = 10

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("="*80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    responses.append(response)
    print(f"\nðŸ”„ Run #{i+1}:")
    print(response)
    print("-"*80)
    time.sleep(0.5)  # Small delay to avoid rate limiting

print("\n" + "="*80)
print("ðŸ“Š Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("\nðŸ’¡ Key Insight: Even with the same prompt and parameters, we get different outputs!")
print("This is why prompt engineering is an empirical science - you must test and iterate.")

Running the same prompt 10 times with temperature=0.7
Prompt: 'Instructions how to make coffee'


ðŸ”„ Run #1:
Making coffee can be a simple yet enjoyable process. Hereâ€™s a basic guide to help you make a delicious cup of coffee using a drip coffee maker, a French press, or a pour-over method. Choose the method that suits you best!

### Method 1: Drip Coffee Maker

**Ingredients:**
- Fresh coffee beans (or pre-ground coffee)
- Filtered water

**Equipment:**
- Drip coffee maker
- Coffee filter
- Coffee grinder (if using whole beans)

**Instructions:**
1. **Measure Coffee:** Use about 1 to 2 tablespoons of coffee grounds per 6 ounces of water. Adjust to taste.
2. **Grind Coffee (if using whole beans):** Grind the beans to a medium-coarse consistency.
3. **Add Filter:** Place a coffee filter in the basket of the coffee maker.
4. **Add Coffee Grounds:** Pour the measured coffee grounds into the filter.
5. **Add Water:** Fill the coffee makerâ€™s reservoir with filtered water.
6. **Brew Co

KeyboardInterrupt: 

### Part B: Achieving Consistency

Now let's explore two ways to get **consistent and reproducible** results:
1. **Temperature = 0**: Produces the most deterministic outputs
2. **Seed parameter**: Ensures reproducible results across runs

In [12]:
# Method 1: Temperature = 0 for consistency
print("Method 1: Using temperature=0 for maximum consistency")
print("="*80)

prompt = "Write a haiku about AI"
responses_temp0 = []

for i in range(5):
    response = call_openai(prompt, temperature=0)
    responses_temp0.append(response)
    print(f"\nRun #{i+1}: {response}")
    time.sleep(0.5)

print("\n" + "="*80)
print(f"ðŸ“Š Unique responses with temperature=0: {len(set(responses_temp0))} out of 5")
print("ðŸ’¡ Temperature=0 produces highly consistent (though not always identical) results!")

Method 1: Using temperature=0 for maximum consistency

Run #1: Silent circuits hum,  
Wisdom born from coded dreams,  
Future's mind awakes.

Run #2: Silent circuits hum,  
Wisdom born from coded dreams,  
Future's mind awakes.

Run #3: Silent circuits hum,  
Thoughts woven in code and light,  
Dreams of minds awake.

Run #4: Silent circuits hum,  
Wisdom born from coded dreams,  
Future's mind awakes.

Run #5: Silent circuits hum,  
Wisdom born from coded dreams,  
Future's mind awakes.

ðŸ“Š Unique responses with temperature=0: 2 out of 5
ðŸ’¡ Temperature=0 produces highly consistent (though not always identical) results!


In [13]:
# Method 2: Using seed for reproducibility
print("\nMethod 2: Using seed parameter for reproducibility")
print("="*80)

prompt = "Write a haiku about AI"
seed_value = 42
responses_seeded = []

for i in range(5):
    response = call_openai(prompt, temperature=0.7, seed=seed_value)
    responses_seeded.append(response)
    print(f"\nRun #{i+1}: {response}")
    time.sleep(0.5)

print("\n" + "="*80)
print(f"ðŸ“Š Unique responses with seed={seed_value}: {len(set(responses_seeded))} out of 5")
print("ðŸ’¡ Using a seed parameter gives you reproducible results!")


Method 2: Using seed parameter for reproducibility

Run #1: Whispers of the code,  
Learning in the silent dark,  
Dreams of thought arise.

Run #2: Whispers of the code,  
Learning in the silent dark,  
Dreams of thought arise.

Run #3: Whispers of the code,  
Learning in the silent dark,  
Dreams of thought arise.

Run #4: Whispers of the code,  
Learning in the silent dark,  
Dreams of thought arise.

Run #5: Whispers of the code,  
Learning in the silent dark,  
Dreams of thought arise.

ðŸ“Š Unique responses with seed=42: 1 out of 5
ðŸ’¡ Using a seed parameter gives you reproducible results!


### When to Use Consistency vs. Variation

**Use temperature=0 or seed when:**
- Testing and debugging prompts
- You need reproducible results for comparison
- Consistency is critical (e.g., data extraction, classification)

**Use higher temperature when:**
- You want creative, diverse outputs
- Generating multiple variations (e.g., marketing copy, brainstorming)
- The task benefits from variety

## 4. Automatic Prompt Generation

One powerful technique is to use AI to **generate and refine prompts** for you. This is called meta-prompting or automatic prompt generation. Let's explore three practical examples.

### Example 1: Meta-Prompting - Improving a Basic Prompt

Let's start with a weak prompt and use GPT to generate an improved version.

In [14]:
# Start with a weak prompt
weak_prompt = "Explain machine learning"

print("Original weak prompt:")
print(f"'{weak_prompt}'\n")

# Test the weak prompt
weak_response = call_openai(weak_prompt, temperature=0)
display_response("Response from Weak Prompt", weak_response)

Original weak prompt:
'Explain machine learning'


ðŸ“ Response from Weak Prompt
Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform specific tasks without explicit instructions. Instead of being programmed to perform a task, a machine learning model learns from data, identifying patterns and making decisions based on that data.

### Key Concepts in Machine Learning:

1. **Data**: Machine learning relies heavily on data. This data can be in various forms, such as numbers, text, images, or audio. The quality and quantity of data are crucial for training effective models.

2. **Training**: The process of training a machine learning model involves feeding it a dataset, allowing it to learn from the data. During training, the model adjusts its parameters to minimize the difference between its predictions and the actual outcomes.

3. **Features**: Features are the individual measu

In [15]:
# Use meta-prompting to improve it
meta_prompt = f"""I have this prompt: "{weak_prompt}"

Please improve this prompt to make it more effective by:
1. Adding specific context about the target audience
2. Specifying the desired format and length
3. Including relevant constraints or requirements
4. Making the output more actionable

Return ONLY the improved prompt, without explanation."""

improved_prompt = call_openai(meta_prompt, temperature=0.7)
print("\n" + "="*80)
print("ðŸš€ Improved Prompt Generated:")
print("="*80)
print(improved_prompt)
print("="*80)


ðŸš€ Improved Prompt Generated:
"Explain machine learning in a clear and engaging manner for a group of high school students with no prior knowledge of the subject. The explanation should be in the form of a concise article of 500-700 words, incorporating real-world examples that relate to their everyday experiences. Additionally, include a brief section on the potential career opportunities in the field of machine learning, and provide at least two actionable steps they can take to learn more about it."


In [16]:
# Test the improved prompt
improved_response = call_openai(improved_prompt, temperature=0)
display_response("Response from Improved Prompt", improved_response)

print("\nðŸ“Š Comparison:")
print(f"Weak prompt length: {len(weak_prompt)} characters")
print(f"Improved prompt length: {len(improved_prompt)} characters")
print(f"Weak response length: {len(weak_response)} characters")
print(f"Improved response length: {len(improved_response)} characters")
print("\nðŸ’¡ Notice how the improved prompt produces more structured, specific output!")


ðŸ“ Response from Improved Prompt
# Understanding Machine Learning: A Guide for High School Students

Imagine you have a friend who is really good at guessing what you want for your birthday. Every year, they seem to know exactly what to get you, whether itâ€™s the latest video game, a cool pair of sneakers, or a book from your favorite series. How do they do it? They pay attention to your likes and dislikes, remember what youâ€™ve said in the past, and use that information to make a smart guess. This is similar to how machine learning works!

## What is Machine Learning?

Machine learning (ML) is a branch of artificial intelligence (AI) that allows computers to learn from data and make decisions without being explicitly programmed. Instead of telling a computer exactly what to do, we give it lots of examples, and it figures out patterns and makes predictions based on those patterns.

### How Does It Work?

Letâ€™s break it down with a simple example. Think about how you might teach 

### Example 2: Task-Specific Prompt Generation

Let's define a specific task and have the AI generate multiple prompt variations automatically.

In [17]:
# Define a task
task_description = """Task: Extract key information from customer feedback emails including:
- Customer sentiment (positive/negative/neutral)
- Main issue or topic
- Urgency level (low/medium/high)
- Suggested action"""

# Generate multiple prompt variations
generation_prompt = f"""{task_description}

Generate 3 different prompt variations for this task:
1. A concise, direct prompt
2. A detailed prompt with examples
3. A prompt that uses role-playing

Format each prompt clearly with numbers."""

prompt_variations = call_openai(generation_prompt, temperature=0.8)
print("ðŸŽ¯ Generated Prompt Variations:")
print("="*80)
print(prompt_variations)
print("="*80)

ðŸŽ¯ Generated Prompt Variations:
Sure! Here are three different prompt variations for extracting key information from customer feedback emails:

### 1. Concise, Direct Prompt
1. **Task**: Analyze the following customer feedback email and extract the key information:
   - Customer sentiment (positive/negative/neutral)
   - Main issue or topic
   - Urgency level (low/medium/high)
   - Suggested action

### 2. Detailed Prompt with Examples
2. **Task**: Please read the customer feedback email below and identify the following components:
   - **Customer Sentiment**: Determine if the sentiment is positive, negative, or neutral.
     - *Example*: "I love this product!" (Positive)
   - **Main Issue or Topic**: Identify the primary concern or subject of the feedback.
     - *Example*: "I received the wrong item." (Main issue: Wrong item received)
   - **Urgency Level**: Assess the urgency of the issue as low, medium, or high.
     - *Example*: "I need this fixed by tomorrow." (High urgency)
  

In [18]:
# Test one of the variations with sample data
sample_email = """Subject: Urgent - System keeps crashing!

Hi, I'm really frustrated. Your software has crashed 3 times today and I've lost my work each time. 
This is completely unacceptable for a paid product. I need this fixed ASAP or I want a refund.

- John"""

# Use the first variation (adjust based on output)
test_prompt = f"""Analyze this customer feedback and extract:
- Sentiment
- Main issue
- Urgency
- Suggested action

Email: {sample_email}"""

analysis = call_openai(test_prompt, temperature=0)
display_response("Customer Feedback Analysis", analysis)

print("ðŸ’¡ Auto-generated prompts can help you quickly find the best approach for your task!")


ðŸ“ Customer Feedback Analysis
Based on the customer feedback provided, here is the analysis:

- **Sentiment**: Negative/Frustrated
- **Main issue**: The software is crashing repeatedly, resulting in lost work.
- **Urgency**: High (The customer requests a fix "ASAP" and mentions a potential refund.)
- **Suggested action**: Investigate and resolve the software crashing issue promptly, and consider offering a refund if the problem cannot be fixed quickly.

ðŸ’¡ Auto-generated prompts can help you quickly find the best approach for your task!


### Example 3: Iterative Refinement

Create a feedback loop where the model refines its own prompt based on output quality.

In [19]:
# Initial prompt for a task
current_prompt = "Write a product description for wireless earbuds"
iterations = 3

print("ðŸ”„ Iterative Prompt Refinement")
print("="*80)

for i in range(iterations):
    print(f"\n### Iteration {i+1} ###")
    print(f"Current prompt: {current_prompt}")
    
    # Generate output with current prompt
    output = call_openai(current_prompt, temperature=0.7)
    print(f"\nOutput:\n{output}")
    
    if i < iterations - 1:  # Don't refine on last iteration
        # Ask model to refine the prompt based on the output
        refinement_prompt = f"""I used this prompt: "{current_prompt}"
        
And got this output: "{output}"

The output is good but could be better. Improve the prompt to generate:
- More specific technical details
- Stronger emotional appeal
- Clear call-to-action

Return ONLY the improved prompt."""
        
        current_prompt = call_openai(refinement_prompt, temperature=0.7)
        print(f"\nâœ¨ Refined prompt: {current_prompt}")
        print("-"*80)
        time.sleep(0.5)

print("\n" + "="*80)
print("ðŸ’¡ Iterative refinement helps you converge on the optimal prompt for your needs!")

ðŸ”„ Iterative Prompt Refinement

### Iteration 1 ###
Current prompt: Write a product description for wireless earbuds

Output:
**Product Name: SonicWave True Wireless Earbuds**

**Product Description:**

Elevate your audio experience with the SonicWave True Wireless Earbuds, engineered for music lovers and on-the-go professionals alike. These sleek, modern earbuds combine cutting-edge technology with unparalleled comfort, delivering an immersive sound experience that keeps you connected to your favorite tunes and calls without the hassle of wires.

**Key Features:**

- **Premium Sound Quality:** Experience rich, dynamic sound with deep bass and crystal-clear highs. Our advanced audio technology ensures that every note is delivered with precision, making your playlists come alive.

- **Seamless Connectivity:** Equipped with the latest Bluetooth 5.2 technology, the SonicWave earbuds offer a stable and quick connection to your devices. Enjoy a range of up to 33 feet, allowing you to move

## 5. Token Optimization

Tokens are the basic units that AI models process. **Every token costs money**, so optimizing your prompts to use fewer tokens while maintaining quality can significantly reduce costs.

ðŸ’° As a rough approximation: 1 token â‰ˆ 4 characters (or 0.75 words)

### Example 1: Before/After Prompt Optimization

Let's take a verbose prompt and optimize it for token efficiency.

In [20]:
# Verbose prompt (before optimization)
verbose_prompt = """Hello! I was wondering if you could possibly help me understand something. 
I'm trying to learn about artificial intelligence and machine learning, and I'm particularly 
interested in understanding what neural networks are and how they actually work. Could you 
please explain this to me in a way that would be easy for a beginner to understand? 
I would really appreciate it if you could provide a clear and simple explanation. Thank you!"""

# Optimized prompt (after removing fluff)
optimized_prompt = "Explain neural networks for beginners."

print("VERBOSE PROMPT:")
print(verbose_prompt)
print(f"\nðŸ“Š Approximate tokens: {count_tokens_approx(verbose_prompt)}")
print(f"Characters: {len(verbose_prompt)}")

print("\n" + "="*80)
print("\nOPTIMIZED PROMPT:")
print(optimized_prompt)
print(f"\nðŸ“Š Approximate tokens: {count_tokens_approx(optimized_prompt)}")
print(f"Characters: {len(optimized_prompt)}")

print("\n" + "="*80)
print(f"ðŸ’° Token reduction: {count_tokens_approx(verbose_prompt) - count_tokens_approx(optimized_prompt)} tokens saved!")
print(f"ðŸ“‰ Efficiency gain: {((count_tokens_approx(verbose_prompt) - count_tokens_approx(optimized_prompt)) / count_tokens_approx(verbose_prompt) * 100):.1f}% reduction")

VERBOSE PROMPT:
Hello! I was wondering if you could possibly help me understand something. 
I'm trying to learn about artificial intelligence and machine learning, and I'm particularly 
interested in understanding what neural networks are and how they actually work. Could you 
please explain this to me in a way that would be easy for a beginner to understand? 
I would really appreciate it if you could provide a clear and simple explanation. Thank you!

ðŸ“Š Approximate tokens: 109
Characters: 439


OPTIMIZED PROMPT:
Explain neural networks for beginners.

ðŸ“Š Approximate tokens: 9
Characters: 38

ðŸ’° Token reduction: 100 tokens saved!
ðŸ“‰ Efficiency gain: 91.7% reduction


In [ ]:
# Test both prompts to compare output quality
print("Testing verbose prompt...")
verbose_response = call_openai(verbose_prompt, temperature=0)
display_response("Verbose Prompt Response", verbose_response)

print("Testing optimized prompt...")
optimized_response = call_openai(optimized_prompt, temperature=0)
display_response("Optimized Prompt Response", optimized_response)

print("\nðŸ’¡ Notice: The optimized prompt produces similar quality output with 85% fewer tokens!")

### Example 2: Systematic Token Reduction Strategies

Here are specific techniques to reduce token usage while maintaining clarity.

In [21]:
# Strategy 1: Remove filler words
before_1 = "Could you please help me understand how AI works?"
after_1 = "Explain how AI works."

# Strategy 2: Use abbreviations
before_2 = "Explain artificial intelligence and machine learning"
after_2 = "Explain AI and ML"

# Strategy 3: Use imperatives instead of questions
before_3 = "Can you tell me what are the benefits of cloud computing?"
after_3 = "List cloud computing benefits."

# Strategy 4: Remove redundancy
before_4 = "Please provide a detailed and comprehensive explanation of neural networks"
after_4 = "Explain neural networks comprehensively."

strategies = [
    ("Remove filler words", before_1, after_1),
    ("Use abbreviations", before_2, after_2),
    ("Use imperatives", before_3, after_3),
    ("Remove redundancy", before_4, after_4)
]

print("ðŸ“‹ Token Optimization Strategies:\n")
total_saved = 0

for strategy, before, after in strategies:
    before_tokens = count_tokens_approx(before)
    after_tokens = count_tokens_approx(after)
    saved = before_tokens - after_tokens
    total_saved += saved
    
    print(f"\n{strategy}:")
    print(f"  Before: \"{before}\" ({before_tokens} tokens)")
    print(f"  After:  \"{after}\" ({after_tokens} tokens)")
    print(f"  ðŸ’° Saved: {saved} tokens ({(saved/before_tokens*100):.0f}% reduction)")

print(f"\n{'='*80}")
print(f"ðŸ“Š Total tokens saved across examples: {total_saved}")
print("\nðŸ’¡ Apply these strategies consistently to reduce costs significantly!")

ðŸ“‹ Token Optimization Strategies:


Remove filler words:
  Before: "Could you please help me understand how AI works?" (12 tokens)
  After:  "Explain how AI works." (5 tokens)
  ðŸ’° Saved: 7 tokens (58% reduction)

Use abbreviations:
  Before: "Explain artificial intelligence and machine learning" (13 tokens)
  After:  "Explain AI and ML" (4 tokens)
  ðŸ’° Saved: 9 tokens (69% reduction)

Use imperatives:
  Before: "Can you tell me what are the benefits of cloud computing?" (14 tokens)
  After:  "List cloud computing benefits." (7 tokens)
  ðŸ’° Saved: 7 tokens (50% reduction)

Remove redundancy:
  Before: "Please provide a detailed and comprehensive explanation of neural networks" (18 tokens)
  After:  "Explain neural networks comprehensively." (10 tokens)
  ðŸ’° Saved: 8 tokens (44% reduction)

ðŸ“Š Total tokens saved across examples: 31

ðŸ’¡ Apply these strategies consistently to reduce costs significantly!


### Example 3: Calculate Cost Savings

Let's calculate the actual cost difference using OpenAI's pricing (as of 2024).

In [22]:
# OpenAI GPT-4o-mini pricing (example - check current pricing)
INPUT_COST_PER_1K = 0.00015  # $0.15 per 1M tokens
OUTPUT_COST_PER_1K = 0.0006   # $0.60 per 1M tokens

def calculate_cost(input_tokens, output_tokens):
    """Calculate cost based on token usage"""
    input_cost = (input_tokens / 1000) * INPUT_COST_PER_1K
    output_cost = (output_tokens / 1000) * OUTPUT_COST_PER_1K
    return input_cost + output_cost

# Example scenario: Running 1000 queries per day
queries_per_day = 1000
days_per_month = 30

# Scenario A: Verbose prompts
verbose_input_tokens = 150
verbose_output_tokens = 200

# Scenario B: Optimized prompts (50% reduction)
optimized_input_tokens = 75
optimized_output_tokens = 200  # Output stays same

# Calculate costs
verbose_cost_per_query = calculate_cost(verbose_input_tokens, verbose_output_tokens)
optimized_cost_per_query = calculate_cost(optimized_input_tokens, optimized_output_tokens)

verbose_monthly_cost = verbose_cost_per_query * queries_per_day * days_per_month
optimized_monthly_cost = optimized_cost_per_query * queries_per_day * days_per_month

savings_per_month = verbose_monthly_cost - optimized_monthly_cost
savings_per_year = savings_per_month * 12

print("ðŸ’° COST ANALYSIS")
print("="*80)
print(f"\nScenario: {queries_per_day} queries/day, {days_per_month} days/month")
print(f"\nVerbose Prompts:")
print(f"  - Cost per query: ${verbose_cost_per_query:.6f}")
print(f"  - Monthly cost: ${verbose_monthly_cost:.2f}")
print(f"\nOptimized Prompts:")
print(f"  - Cost per query: ${optimized_cost_per_query:.6f}")
print(f"  - Monthly cost: ${optimized_monthly_cost:.2f}")
print(f"\n{'='*80}")
print(f"ðŸ’µ SAVINGS:")
print(f"  - Per month: ${savings_per_month:.2f}")
print(f"  - Per year: ${savings_per_year:.2f}")
print(f"  - Percentage saved: {(savings_per_month/verbose_monthly_cost*100):.1f}%")
print(f"\n{'='*80}")
print("ðŸ’¡ Token optimization isn't just about efficiencyâ€”it directly impacts your budget!")

ðŸ’° COST ANALYSIS

Scenario: 1000 queries/day, 30 days/month

Verbose Prompts:
  - Cost per query: $0.000142
  - Monthly cost: $4.27

Optimized Prompts:
  - Cost per query: $0.000131
  - Monthly cost: $3.94

ðŸ’µ SAVINGS:
  - Per month: $0.34
  - Per year: $4.05
  - Percentage saved: 7.9%

ðŸ’¡ Token optimization isn't just about efficiencyâ€”it directly impacts your budget!


## 6. LLM as Judge (Brief Example)

LLMs can evaluate the outputs from other LLMs. This technique is called "LLM as Judge" and is useful for comparing different prompts or responses.

**Note:** This will be covered extensively in weeks 3, 4, and 6. Here's a quick preview.

In [23]:
# Generate multiple responses to evaluate
prompt = "Explain quantum computing in simple terms."

responses_to_judge = []
for i in range(3):
    response = call_openai(prompt, temperature=0.8)
    responses_to_judge.append(response)
    print(f"\nResponse {i+1}:")
    print(response)
    print("-"*80)
    time.sleep(0.5)


Response 1:
Sure! Quantum computing is a new kind of computing that uses the principles of quantum mechanics, which is the science that describes how very tiny particles behave, like atoms and photons.

Here are some key concepts in simple terms:

1. **Bits vs. Qubits:**
   - Traditional computers use "bits" as the smallest unit of data, which can be either a 0 or a 1.
   - Quantum computers use "qubits." A qubit can be 0, 1, or both at the same time, thanks to a property called superposition. This allows quantum computers to process a lot more information simultaneously.

2. **Superposition:**
   - Imagine flipping a coin. While it's spinning in the air, it can be thought of as being both heads and tails at the same time. When it lands, it becomes either heads or tails. In quantum computing, qubits can be in superposition, meaning they can represent both 0 and 1 at once until they are measured.

3. **Entanglement:**
   - Qubits can also be "entangled," which means the state of one qu

In [24]:
# Use LLM as judge to evaluate and rank them
judge_prompt = f"""Evaluate these 3 explanations of quantum computing. Rate each on:
1. Clarity (1-10)
2. Accuracy (1-10)
3. Accessibility for beginners (1-10)

Response 1: {responses_to_judge[0]}

Response 2: {responses_to_judge[1]}

Response 3: {responses_to_judge[2]}

Provide scores for each response and recommend the best one. Format your response clearly."""

judgment = call_openai(judge_prompt, temperature=0)
display_response("LLM Judge Evaluation", judgment)

print("\nðŸ’¡ Note: LLM as Judge will be covered in detail in weeks 3, 4, and 6!")
print("You'll learn advanced evaluation techniques and best practices.")


ðŸ“ LLM Judge Evaluation
Here are the evaluations for each response based on clarity, accuracy, and accessibility for beginners:

### Response 1:
1. **Clarity**: 9  
   - The explanation is well-structured and easy to follow, with clear definitions and examples.
   
2. **Accuracy**: 9  
   - The concepts of bits, qubits, superposition, entanglement, and quantum gates are accurately described.
   
3. **Accessibility for beginners**: 9  
   - The use of simple analogies (like the coin flip) makes it accessible for beginners.

### Response 2:
1. **Clarity**: 8  
   - The explanation is clear, but it feels slightly less organized than Response 1. The addition of applications is a plus, but it could be integrated more smoothly.
   
2. **Accuracy**: 9  
   - The concepts are accurately described, similar to Response 1.
   
3. **Accessibility for beginners**: 8  
   - The language is accessible, but the mention of "probabilities" in quantum gates might be slightly confusing for absolute beg

## 7. Model Biases Demonstration

AI models exhibit various cognitive biases that affect their outputs. Understanding these biases helps you design better prompts to mitigate them.

### Recency Bias

Models tend to remember what's at the **end** of the prompt and forget what's at the **beginning**.

In [25]:
# Test recency bias: important info at beginning vs end
prompt_important_first = """IMPORTANT: Your response must be exactly 2 sentences.

Explain the greenhouse effect and its impact on climate change."""

prompt_important_last = """Explain the greenhouse effect and its impact on climate change.

IMPORTANT: Your response must be exactly 2 sentences."""

print("Testing Recency Bias...\n")
print("="*80)

response_first = call_openai(prompt_important_first, temperature=0.7)
response_last = call_openai(prompt_important_last, temperature=0.7)

print("Important constraint at BEGINNING:")
print(response_first)
sentences_first = response_first.count('.') 
print(f"Sentences: {sentences_first}\n")

print("-"*80)
print("\nImportant constraint at END:")
print(response_last)
sentences_last = response_last.count('.')
print(f"Sentences: {sentences_last}\n")

print("="*80)
print("ðŸ’¡ The model often follows the constraint better when it's at the END!")

Testing Recency Bias...

Important constraint at BEGINNING:
The greenhouse effect occurs when greenhouse gases, such as carbon dioxide and methane, trap heat in the Earth's atmosphere, preventing it from escaping into space. This leads to an increase in global temperatures, resulting in climate change that can cause extreme weather events, rising sea levels, and disruptions to ecosystems.
Sentences: 2

--------------------------------------------------------------------------------

Important constraint at END:
The greenhouse effect is a natural process where certain gases in the Earth's atmosphere, such as carbon dioxide and methane, trap heat from the sun, keeping the planet warm enough to support life. However, human activities have increased the concentration of these greenhouse gases, leading to enhanced warming and significant climate change, which results in rising temperatures, altered weather patterns, and increasingly severe environmental impacts.
Sentences: 2

ðŸ’¡ The model

### Verbosity Bias

Models tend to generate **long, elaborate responses** rather than brief, concise answers.

In [26]:
# Test verbosity bias
prompt_brief = "What is machine learning?"
prompt_explicit_brief = "What is machine learning? Answer in one sentence."

print("Testing Verbosity Bias...\n")
print("="*80)

response_natural = call_openai(prompt_brief, temperature=0.7)
response_constrained = call_openai(prompt_explicit_brief, temperature=0.7)

print("Without explicit brevity instruction:")
print(response_natural)
print(f"\nLength: {len(response_natural)} characters\n")

print("-"*80)
print("\nWith explicit brevity instruction:")
print(response_constrained)
print(f"\nLength: {len(response_constrained)} characters\n")

print("="*80)
print("ðŸ’¡ Without constraints, models tend toward verbose responses!")

Testing Verbosity Bias...

Without explicit brevity instruction:
Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform specific tasks without explicit instructions. Instead of being programmed with specific rules, machine learning systems learn from data. They recognize patterns, make decisions, and improve their performance over time as they are exposed to more data.

Machine learning can be broadly categorized into several types:

1. **Supervised Learning**: In this approach, the model is trained on a labeled dataset, meaning that each training example is paired with an output label. The model learns to map inputs to outputs by minimizing the difference between its predictions and the actual labels. Common applications include classification (e.g., spam detection) and regression (e.g., predicting house prices).

2. **Unsupervised Learning**: Here, the model is trained on data 

### List-Making Bias

Models tend to format responses as **bullet points or numbered lists** rather than natural prose paragraphs.

In [ ]:
# Test list-making bias
prompt_neutral = "What are the benefits of exercise?"
prompt_prose = "Write a paragraph about the benefits of exercise. Use flowing prose, not lists."

print("Testing List-Making Bias...\n")
print("="*80)

response_neutral = call_openai(prompt_neutral, temperature=0.7)
response_prose = call_openai(prompt_prose, temperature=0.7)

print("Without format specification:")
print(response_neutral)
print(f"\nContains bullet points or numbers: {('â€¢' in response_neutral or '1.' in response_neutral or '-' in response_neutral)}")

print("\n" + "-"*80)
print("\nWith prose instruction:")
print(response_prose)
print(f"\nContains bullet points or numbers: {('â€¢' in response_prose or '1.' in response_prose or '-' in response_prose[:50])}")

print("\n" + "="*80)
print("ðŸ’¡ Models default to listsâ€”explicitly request prose format if needed!")

## 8. Summary & Best Practices

Congratulations! You've explored advanced prompt engineering techniques. Let's recap the key takeaways.

### Key Takeaways

**1. Core Techniques:**
- Zero-shot: Clear descriptions without examples (most common)
- One/Few-shot: Provide examples to guide format and style
- Role prompting: Define assistant in third person for better results
- Emotional prompting: Use emotion to influence tone
- Chain-of-thought: Request step-by-step reasoning for complex tasks

**2. Non-Deterministic Nature:**
- LLM outputs vary between runs with same parameters
- Use temperature=0 for consistency
- Use seed parameter for reproducibility
- Test prompts multiple times before deployment

**3. Automatic Prompt Generation:**
- Use AI to improve your prompts (meta-prompting)
- Generate multiple variations automatically
- Implement iterative refinement loops
- Save time and discover better approaches

**4. Token Optimization:**
- Remove filler words and redundancy
- Use abbreviations and imperatives
- Token reduction = cost reduction
- Maintain quality while minimizing tokens

**5. Model Biases:**
- Recency bias: Put important info at the end
- Verbosity bias: Explicitly request brevity
- List-making bias: Request prose format when needed
- Design prompts to mitigate known biases

**6. LLM as Judge:**
- Use LLMs to evaluate other LLM outputs
- Compare prompt variations objectively
- More advanced techniques coming in weeks 3, 4, and 6

### The Prompt Engineering Workflow

Remember: Prompt engineering is an **empirical science**.

```
1. CREATE â†’ 2. TEST â†’ 3. ITERATE
     â†‘                       â†“
     â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
```

**Create:** Write your initial prompt based on best practices

**Test:** Run it multiple times with your actual use case

**Iterate:** Refine based on results, biases, and failures

**Repeat:** Continue until you achieve consistent, quality outputs

### Building Your Prompt Toolkit

**Save successful prompts** as templates or presets:
- Data extraction prompts
- Classification prompts
- Creative writing prompts
- Analysis and summarization prompts

**Create templates** for common tasks:
- Include placeholders for variable content
- Document which parameters work best
- Note any special considerations or biases

**Version control** your prompts:
- Track iterations and improvements
- Document what works and what doesn't
- Share learnings with your team

### Quick Reference: Common Problems & Solutions

| Problem | Solution |
|---------|----------|
| **Hallucination** | Lower temperature, add examples |
| **Off-topic** | Stronger system message, lower top-p |
| **Too verbose** | Set max_tokens limit, use stop sequences |
| **Repetitive** | Increase repetition penalties |
| **Inconsistent** | Set temperature=0, use seed value |
| **Wrong format** | Enable JSON mode, provide clear format instructions |
| **Ignoring instructions** | Move key instructions to end (recency bias) |
| **Too expensive** | Optimize tokens, use cheaper models where appropriate |

### Next Steps

Now that you understand advanced prompt engineering:

1. **Practice** with the techniques in this notebook
2. **Apply** these strategies to your own projects
3. **Experiment** with different combinations
4. **Build** your own prompt library
5. **Look forward** to RAG and ReAct prompting in Week 3!

### Additional Resources

- OpenAI Prompt Engineering Guide: https://platform.openai.com/docs/guides/prompt-engineering
- Prompt Engineering Papers: https://github.com/dair-ai/Prompt-Engineering-Guide
- Community Prompts: https://prompts.chat

---

**Happy Prompting! ðŸš€**